In [ ]:
import json
from pathlib import Path

import joblib
import pandas as pd
import numpy as np
from scipy.stats import binomtest

In [ ]:
echonext_dirs = {
    72475: "runs-echonext",
    32768: "runs-echonext-32k",
    16384: "runs-echonext-16k",
    8192: "runs-echonext-8k",
    4096: "runs-echonext-4k",
    2048: "runs-echonext-2k",
    1024: "runs-echonext-1k",
    512: "runs-echonext-512",
    256: "runs-echonext-256",
}

### LAP vs ProtoPool Runtime

In [ ]:
experiments = {
    # nodes we ran on have different relative speeds so only use runs that entirely executed on a single node
    "ProtoSSL HEEDB": "/opt/gpu_working/steven/new/protossl-outputs-seed{seed}/{run_dir}/protossl-heedb-pila",
    "ProtoSSL HEEDB (PIA)": "/opt/gpu_working/steven/new/protossl-outputs-seed{seed}/{run_dir}/protossl-heedb-pia",
}

data = []
for seed in [42, 67, 70, 73, 99]:
    for subset_size, run_dir in echonext_dirs.items():
        for exp_name, exp_dir in experiments.items():
            model_dir = Path(exp_dir.format(seed=seed, run_dir=run_dir))
            summary_json = model_dir / "learn-prototype-assignments/latest/wandb/latest-run/files/wandb-summary.json"
            with open(summary_json, "r") as f:
                summary = json.load(f)
            metadata_json = model_dir / "learn-prototype-assignments/latest/wandb/latest-run/files/wandb-metadata.json"
            with open(metadata_json, "r") as f:
                metadata = json.load(f)

            host = metadata["host"]
            runtime_sec = summary["_runtime"]
            datum = {
                "Host": host,
                "Seed": seed,
                "Train Size": subset_size,
                "Model": exp_name,
                "Runtime": runtime_sec,
            }
            data.append(datum)
data = pd.DataFrame(data)

In [ ]:
df = data.pivot(columns=["Train Size"], index=["Model", "Seed"], values="Runtime").sort_index(axis=1, ascending=False)
df = df.groupby(level=0).mean().map(lambda x: f"{x:0.1f}") + " ± " + df.groupby(level=0).std().map(lambda x: f"{x:0.1f}")

In [ ]:
print(df.to_latex())

### SK-OT convergence

In [ ]:
experiments = {
    # nodes we ran on have different relative speeds so only use runs that entirely executed on a single node
    "ProtoSSL HEEDB": "/opt/gpu_working/steven/protossl-outputs-seed{seed}/{run_dir}/protossl-heedb-pila",
    "ProtoSSL HEEDB (PIA)": "/opt/gpu_working/steven/protossl-outputs-seed{seed}/{run_dir}/protossl-heedb-pia",
}

data = []
for subset_size, run_dir in echonext_dirs.items():
    for seed in [42, 67, 70, 73, 99]:
        model_dir = Path(f"/opt/gpu_working/steven/protossl-outputs-seed{seed}/{run_dir}/ecgfounder-clustering")
        temp = np.load(model_dir / "assign_frec.npy")

        datum = {
            "Seed": seed,
            "Train Size": subset_size,
            "% Still Random": (temp == 0).sum() / len(temp.flatten()),
        }

        data.append(datum)
data = pd.DataFrame(data)

In [ ]:
df = data.pivot(columns=["Train Size"], index=["Seed"], values="% Still Random").sort_index(axis=1, ascending=False)
df

### Positive Prototype Coefficients

In [ ]:
data = []
for seed in [42, 67, 70, 73, 99]:
    for subset_size, run_dir in echonext_dirs.items():
        models = joblib.load(
            f"/opt/gpu_working/steven/new/protossl-outputs-seed{seed}/{run_dir}/protossl-heedb-pila/model.joblib"
        )

        K = 14
        L = len(models.estimators_)
        P = models.estimators_[0].coef_.shape[1]
        assert P % K == 0
        assert P // K == L

        ratios = []
        on_tgt = []
        off_tgt = []
        mask = np.zeros(P, dtype=bool)
        for i, lr in enumerate(models.estimators_):
            coef = lr.coef_
            assert coef.shape[1] == P
            odds = np.exp(coef)
            _mask = mask.copy()
            _mask[i * K : (i + 1) * K] = True
            on = odds[:, _mask].mean()
            off = odds[:, ~_mask].mean()
            on_tgt.append(on)
            off_tgt.append(off)
            ratios.append(on / off)
        data.append(
            {
                "Seed": seed,
                "Train Size": subset_size,
                "On-Target": np.asarray(on_tgt).mean(),
                "Off-Target": np.asarray(off_tgt).mean(),
                "Pos/Neg-Prototype Ratio": np.asarray(ratios).mean(),
            }
        )
df = pd.DataFrame(data)

In [ ]:
temp = df.groupby("Train Size")[["Pos/Neg-Prototype Ratio", "On-Target"]].mean().T.sort_index(axis=1, ascending=False)
temp.loc["Ratio > 1"] = ["\\cmark" if x > 1 else "\\xmark" for x in temp.loc["Pos/Neg-Prototype Ratio"]]
temp.loc["Pos-Prototype OR > 1"] = ["\\cmark" if x > 1 else "\\xmark" for x in temp.loc["On-Target"]]
print(temp.loc[["Pos-Prototype OR > 1", "Ratio > 1", "Pos/Neg-Prototype Ratio"]].to_latex(float_format="%.3f"))

In [ ]:
n_above = (df["Pos/Neg-Prototype Ratio"] > 1).sum()
n_total = len(df)
result = binomtest(n_above, n_total, p=0.5, alternative="greater")
print(f"Pos/Neg-Prototype Ratio > 1 in {n_above}/{n_total} cells (sign test p = {result.pvalue:.2e})")